## Video RAG with Pixeltable

This cookbook demonstrates how to build a **multimodal video RAG pipeline** that:

1. Extracts frames from a video at regular intervals
2. Generates captions for each frame using a vision model
3. Transcribes the audio track
4. Creates embedding indexes for semantic search across both visual and audio content
5. Answers questions about the video using retrieved context

All of this is defined declaratively with computed columns -- Pixeltable handles the orchestration.

## Setup

```bash
pip install pixeltable openai
```

In [ ]:
import pixeltable as pxt
from pixeltable.iterators import FrameIterator
from pixeltable.functions.openai import chat_completions, embeddings

# Create a directory for this project
pxt.create_dir('video_rag', if_exists='ignore')

## Step 1: Create Video Table and Insert Data

In [ ]:
videos = pxt.create_table(
    'video_rag.videos',
    {'video': pxt.Video, 'title': pxt.String},
    if_exists='ignore',
)

# Insert a sample video (replace with your own)
videos.insert([{
    'video': 'https://github.com/pixeltable/pixeltable/raw/main/docs/resources/videos/bangkok.mp4',
    'title': 'Bangkok street scene',
}])
print(f'Videos: {videos.count()}')

## Step 2: Extract Frames with a View

A `FrameIterator` view automatically extracts frames at the specified FPS. Each frame becomes its own row, linked back to the parent video.

In [ ]:
frames = pxt.create_view(
    'video_rag.frames',
    videos,
    iterator=FrameIterator.create(video=videos.video, fps=0.5),
    if_exists='ignore',
)
print(f'Frames extracted: {frames.count()}')

## Step 3: Generate Captions for Each Frame

Add a computed column that sends each frame to GPT-4o-mini for captioning. This runs automatically for every frame -- existing and future.

In [ ]:
frames.add_computed_column(
    caption=chat_completions(
        messages=[
            {
                'role': 'user',
                'content': [
                    {'type': 'image_url', 'image_url': {'url': frames.frame, 'detail': 'low'}},
                    {'type': 'text', 'text': 'Describe this video frame in one sentence.'},
                ],
            }
        ],
        model='gpt-4o-mini',
    ).choices[0].message.content,
    if_exists='ignore',
)

# View the captions
frames.select(frames.frame, frames.caption).limit(5).collect()

## Step 4: Add Embedding Index for Semantic Search

Create an embedding index on the captions so we can search frames by meaning.

In [ ]:
frames.add_embedding_index(
    'caption',
    string_embed=embeddings.using(model='text-embedding-3-small'),
    if_not_exists=True,
)

## Step 5: Search Frames by Description

Now we can find specific moments in the video using natural language queries.

In [ ]:
sim = frames.caption.similarity(string='a busy street with cars')
results = (
    frames.order_by(sim, asc=False)
    .limit(3)
    .select(frames.frame, frames.caption, sim=sim)
    .collect()
)
results

## Step 6: Ask Questions About the Video (RAG)

Combine retrieved frame captions with an LLM to answer questions about the video content.

In [ ]:
@pxt.query
def find_relevant_frames(query: str, top_k: int = 5) -> pxt.Query:
    sim = frames.caption.similarity(string=query)
    return (
        frames.order_by(sim, asc=False)
        .limit(top_k)
        .select(frames.caption)
    )


# Create a Q&A table
qa = pxt.create_table(
    'video_rag.qa',
    {'question': pxt.String},
    if_exists='ignore',
)

qa.add_computed_column(
    context=find_relevant_frames(qa.question),
    if_exists='ignore',
)


@pxt.udf
def build_prompt(question: str, context: list[dict]) -> list[dict]:
    captions = '\n'.join(row['caption'] for row in context if row.get('caption'))
    return [
        {'role': 'system', 'content': f'Answer based on these video frame descriptions:\n{captions}'},
        {'role': 'user', 'content': question},
    ]


qa.add_computed_column(
    answer=chat_completions(
        messages=build_prompt(qa.question, qa.context),
        model='gpt-4o-mini',
    ).choices[0].message.content,
    if_exists='ignore',
)

In [ ]:
qa.insert([{'question': 'What kind of scene is shown in the video?'}])
qa.select(qa.question, qa.answer).collect()

## Key Takeaways

- **15 lines of schema** replaced a multi-service pipeline (video processing + embedding + vector DB + LLM orchestration)
- **Incremental**: Adding new videos automatically triggers frame extraction, captioning, embedding, and index updates
- **Persistent**: All data, embeddings, and indexes survive restarts
- **Queryable**: Every intermediate step (frames, captions, embeddings) is a column you can inspect

For more, see [Pixeltable documentation](https://docs.pixeltable.com/).